# Community Detection Algorithms

In [1]:
from dotenv import load_dotenv
from neo4j_viz.gds import from_gds
from graphdatascience import GraphDataScience
import os

load_dotenv()

True

# Data

In [86]:
URI = "neo4j://localhost:7687"
auth = ("neo4j", os.getenv('NEO4J_AUTH', default='neo4j'))
db = os.getenv('NEO4J_DB', 'neo4j')

gds = GraphDataScience(URI, auth=auth, database=db)

# Check the installed GDS version on the server
print(gds.server_version())

_merge_data_cypher =   """
MERGE (six: Library {name: "six"})
MERGE (pandas: Library {name: "pandas"})
MERGE (numpy: Library {name: "numpy"})
MERGE (python_dateutil: Library {name: "python_dateutil"})
MERGE (pytz: Library {name: "pytz"})
MERGE (pyspark: Library {name: "pyspark"})
MERGE (matplotlib: Library {name: "matplotlib"})
MERGE (spacy: Library {name: "spacy"})
MERGE (py4j: Library {name: "py4j"})
MERGE (jupyter: Library {name: "jupyter"})
MERGE (jpy_console: Library {name: "jpy_console"})
MERGE (nbconvert: Library {name: "nbconvert"})
MERGE (ipykernel: Library {name: "ipykernel"})
MERGE (jpy_client: Library {name: "jpy_client"})
MERGE (jpy_core: Library {name: "jpy_core"})

MERGE (pandas)-[:DEPENDS_ON]->(numpy)
MERGE (pandas)-[:DEPENDS_ON]->(pytz)
MERGE (pandas)-[:DEPENDS_ON]->(python_dateutil)
MERGE (python_dateutil)-[:DEPENDS_ON]->(six)
MERGE (pyspark)-[:DEPENDS_ON]->(py4j)
MERGE (matplotlib)-[:DEPENDS_ON]->(numpy)
MERGE (matplotlib)-[:DEPENDS_ON]->(python_dateutil)
MERGE (matplotlib)-[:DEPENDS_ON]->(six)
MERGE (matplotlib)-[:DEPENDS_ON]->(pytz)
MERGE (spacy)-[:DEPENDS_ON]->(six)
MERGE (spacy)-[:DEPENDS_ON]->(numpy)
MERGE (jupyter)-[:DEPENDS_ON]->(nbconvert)
MERGE (jupyter)-[:DEPENDS_ON]->(ipykernel)
MERGE (jupyter)-[:DEPENDS_ON]->(jpy_console)
MERGE (jpy_console)-[:DEPENDS_ON]->(jpy_client)
MERGE (jpy_console)-[:DEPENDS_ON]->(ipykernel)
MERGE (jpy_client)-[:DEPENDS_ON]->(jpy_core)
MERGE (nbconvert)-[:DEPENDS_ON]->(jpy_core)
  """

gds.run_cypher(_merge_data_cypher)

_graph_name = 'library-deps'
if gds.graph.exists(_graph_name) is not None:
  gds.graph.drop(_graph_name)

# Project the graph into the GDS Graph Catalog
# We call the object representing the projected graph `G_office`
G, project_result = gds.graph.project(
    _graph_name,
    "Library", 
    "DEPENDS_ON")

print('=== project_result\n', project_result)
G

2.13.9
=== project_result
 nodeProjection            {'Library': {'properties': {}, 'label': 'Libra...
relationshipProjection    {'DEPENDS_ON': {'orientation': 'NATURAL', 'agg...
graphName                                                      library-deps
nodeCount                                                                15
relationshipCount                                                        18
projectMillis                                                            14
Name: 0, dtype: object


Graph({'graphName': 'library-deps', 'nodeCount': 15, 'relationshipCount': 18, 'database': 'neo4j', 'configuration': {'relationshipProjection': {'DEPENDS_ON': {'orientation': 'NATURAL', 'aggregation': 'DEFAULT', 'type': 'DEPENDS_ON', 'properties': {}, 'indexInverse': False}}, 'jobId': '39622836-5179-4930-b3a9-2c84d84e4a69', 'nodeProjection': {'Library': {'properties': {}, 'label': 'Library'}}, 'relationshipProperties': {}, 'validateRelationships': False, 'logProgress': True, 'readConcurrency': 4, 'sudo': False, 'nodeProperties': {}}, 'schema': {'graphProperties': {}, 'relationships': {'DEPENDS_ON': {}}, 'nodes': {'Library': {}}}, 'memoryUsage': '290 KiB'})

# Triangle Count and Clustering Coefficient

In [ ]:
# https://neo4j.com/docs/graph-data-science/current/management-ops/graph-update/to-undirected/
gds.graph.relationships.toUndirected(G,relationship_type='DEPENDS_ON',mutate_relationship_type='DEPENDS')

inputRelationships                                                     18
relationshipsWritten                                                   36
mutateMillis                                                            0
postProcessingMillis                                                    0
preProcessingMillis                                                     0
computeMillis                                                          15
configuration           {'jobId': '1784f26f-54ee-4db1-b869-80780f5cfb1...
Name: 0, dtype: object

In [68]:
VG = from_gds(gds, G, db_node_properties=['name'])
# VG.render()

In [ ]:
_res = gds.triangles(G, relationshipTypes=['DEPENDS'])
_res.map(lambda nodeId: gds.util.asNode(nodeId)['name'])

,nodeA,nodeB,nodeC
0,six,python_dateutil,matplotlib
1,jupyter,jpy_console,ipykernel


In [66]:
_res = gds.triangleCount.stream(G, relationshipTypes=['DEPENDS'])
_res['nodeName'] = _res['nodeId'].map(lambda nodeId: gds.util.asNode(nodeId)['name'])
_res

,nodeId,triangleCount,nodeName
0,26,1,six
1,27,0,pandas
2,28,0,numpy
3,29,1,python_dateutil
4,30,0,pytz
5,31,0,pyspark
6,32,1,matplotlib
7,33,0,spacy
8,34,0,py4j
9,35,1,jupyter


# Strongly Connected Components

In [76]:
_res = gds.scc.stream(G, nodeLabels=['Library'], relationshipTypes=['DEPENDS_ON'])
_res['nodeName'] = _res['nodeId'].map(lambda nodeId: gds.util.asNode(nodeId)['name'])
_res

,nodeId,componentId,nodeName
0,26,0,six
1,27,1,pandas
2,28,2,numpy
3,29,3,python_dateutil
4,30,4,pytz
5,31,5,pyspark
6,32,6,matplotlib
7,33,7,spacy
8,34,8,py4j
9,35,9,jupyter


In [88]:
gds.run_cypher("""
MATCH (py4j:Library {name: "py4j"})
MATCH (pyspark:Library {name: "pyspark"})

MERGE (extra:Library {name: "extra"})
MERGE (py4j)-[:DEPENDS_ON]->(extra)
MERGE (extra)-[:DEPENDS_ON]->(pyspark)
""")

_graph_name = 'library-deps'
if gds.graph.exists(_graph_name) is not None:
  gds.graph.drop(_graph_name)
G, project_result = gds.graph.project(
    _graph_name,
    "Library", 
    "DEPENDS_ON")

print('=== project_result\n', project_result)
G

=== project_result
 nodeProjection            {'Library': {'properties': {}, 'label': 'Libra...
relationshipProjection    {'DEPENDS_ON': {'orientation': 'NATURAL', 'agg...
graphName                                                      library-deps
nodeCount                                                                16
relationshipCount                                                        20
projectMillis                                                            12
Name: 0, dtype: object


Graph({'graphName': 'library-deps', 'nodeCount': 16, 'relationshipCount': 20, 'database': 'neo4j', 'configuration': {'relationshipProjection': {'DEPENDS_ON': {'orientation': 'NATURAL', 'aggregation': 'DEFAULT', 'type': 'DEPENDS_ON', 'properties': {}, 'indexInverse': False}}, 'jobId': '7ea38c03-1c56-42ff-8077-f1ad3bc15a4f', 'nodeProjection': {'Library': {'properties': {}, 'label': 'Library'}}, 'relationshipProperties': {}, 'validateRelationships': False, 'logProgress': True, 'readConcurrency': 4, 'sudo': False, 'nodeProperties': {}}, 'schema': {'graphProperties': {}, 'relationships': {'DEPENDS_ON': {}}, 'nodes': {'Library': {}}}, 'memoryUsage': '290 KiB'})

In [90]:
# VG = from_gds(gds, G, db_node_properties=['name'])
# VG.render()

In [92]:
_res = gds.scc.stream(G, nodeLabels=['Library'], relationshipTypes=['DEPENDS_ON'])
_res['nodeName'] = _res['nodeId'].map(lambda nodeId: gds.util.asNode(nodeId)['name'])
_res.sort_values('componentId')

,nodeId,componentId,nodeName
0,42,0,six
1,43,1,pandas
2,44,2,numpy
3,45,3,python_dateutil
4,46,4,pytz
5,47,5,pyspark
15,57,5,extra
8,50,5,py4j
6,48,6,matplotlib
7,49,7,spacy


In [93]:
gds.run_cypher("""
MATCH (extra:Library {name: "extra"})
DETACH DELETE extra
""")

""


In [101]:
_graph_name = 'library-deps'
if gds.graph.exists(_graph_name) is not None:
  gds.graph.drop(_graph_name)
G, project_result = gds.graph.project(
    _graph_name,
    "Library", 
    "DEPENDS_ON")

print('=== project_result\n', project_result)
G

=== project_result
 nodeProjection            {'Library': {'properties': {}, 'label': 'Libra...
relationshipProjection    {'DEPENDS_ON': {'orientation': 'NATURAL', 'agg...
graphName                                                      library-deps
nodeCount                                                                15
relationshipCount                                                        18
projectMillis                                                            16
Name: 0, dtype: object


Graph({'graphName': 'library-deps', 'nodeCount': 15, 'relationshipCount': 18, 'database': 'neo4j', 'configuration': {'relationshipProjection': {'DEPENDS_ON': {'orientation': 'NATURAL', 'aggregation': 'DEFAULT', 'type': 'DEPENDS_ON', 'properties': {}, 'indexInverse': False}}, 'jobId': '8c6b96ba-9f83-44e0-a1a6-b253aa62c50e', 'nodeProjection': {'Library': {'properties': {}, 'label': 'Library'}}, 'relationshipProperties': {}, 'validateRelationships': False, 'logProgress': True, 'readConcurrency': 4, 'sudo': False, 'nodeProperties': {}}, 'schema': {'graphProperties': {}, 'relationships': {'DEPENDS_ON': {}}, 'nodes': {'Library': {}}}, 'memoryUsage': '290 KiB'})

# Connected Components

In [103]:
_res = gds.wcc.stream(G, nodeLabels=['Library'], relationshipTypes=['DEPENDS_ON'])
_res['nodeName'] = _res['nodeId'].map(lambda nodeId: gds.util.asNode(nodeId)['name'])
_res.sort_values('componentId')

,nodeId,componentId,nodeName
0,42,0,six
1,43,0,pandas
2,44,0,numpy
3,45,0,python_dateutil
4,46,0,pytz
6,48,0,matplotlib
7,49,0,spacy
5,47,5,pyspark
8,50,5,py4j
9,51,9,jupyter


# Label Propagation

In [120]:
_res = gds.labelPropagation.stream(G, nodeLabels=['Library'], relationshipTypes=['DEPENDS_ON'], maxIterations=10)
_res['nodeName'] = _res['nodeId'].map(lambda nodeId: gds.util.asNode(nodeId)['name'])
_res.sort_values('communityId')

,nodeId,communityId,nodeName
0,42,42,six
1,43,42,pandas
3,45,42,python_dateutil
7,49,42,spacy
6,48,42,matplotlib
2,44,44,numpy
4,46,46,pytz
5,47,50,pyspark
8,50,50,py4j
9,51,54,jupyter


In [122]:
from neo4j_viz.pandas import from_dfs
_res['source'] = _res['nodeName']
_res['target'] = _res['communityId']
# del _res['nodeName']
# del _res['communityId']
VG = from_dfs(node_dfs=None, rel_dfs=_res)
# VG.render()

# Louvain Modularity

In [126]:
_res = gds.louvain.stream(G, nodeLabels=['Library'], relationshipTypes=['DEPENDS_ON'], maxIterations=10, includeIntermediateCommunities=True)
_res['nodeName'] = _res['nodeId'].map(lambda nodeId: gds.util.asNode(nodeId)['name'])
_res.sort_values('communityId')

,nodeId,communityId,intermediateCommunityIds,nodeName
0,42,2,"[0, 2]",six
1,43,2,"[4, 2]",pandas
2,44,2,"[2, 2]",numpy
3,45,2,"[0, 2]",python_dateutil
4,46,2,"[4, 2]",pytz
6,48,2,"[0, 2]",matplotlib
7,49,2,"[2, 2]",spacy
5,47,8,"[8, 8]",pyspark
8,50,8,"[8, 8]",py4j
9,51,12,"[14, 12]",jupyter


In [128]:
from neo4j_viz.pandas import from_dfs
_res['source'] = _res['nodeName']
_res['target'] = _res['communityId']
# del _res['nodeName']
# del _res['communityId']
VG = from_dfs(node_dfs=None, rel_dfs=_res)
# VG.render()